In [21]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark import SparkContext, SparkConf
import re

def normalize_words(text):
    return re.compile(r'\W+',re.UNICODE).split(text.lower())

## 1. `SaprkContext` 생성

In [22]:
spark_conf = (SparkConf()
    .setMaster('local[1]')
    .setAppName('17')
    .set('spark.driver.host', '127.0.0.1')
    .set('spark.driver.bindAddress', '127.0.0.1')
    .set('spark.pyspark.python', sys.executable)
    .set('spark.python.worker.reuse', 'false')
    .set('spark.ui.enabled', 'false'))

In [23]:
spark = SparkContext.getOrCreate(conf=spark_conf)

## 2. **Load Data** - `RDD`
- `SparkContext.textFile()`: text file을 'RDD'로 로드함
- `SparkContext.parallelize()`: 인자로 전달된 데이터를 `RDD`로 변환함

In [27]:
data_path = r'C:\datas\3rd_week_data\u.data'

# Windows 로컬 환경에서는 큰 파티션의 일부만 가져올 때 Python worker 연결이
# 종료될 수 있으므로, 입력 파일을 작은 파티션으로 나누어 로드합니다.
lines = spark.textFile(data_path, minPartitions=100)

- `lines`: **userID** \t **movieID** \t **rating** \t **timestamp**

In [28]:
print(type(lines))

<class 'pyspark.rdd.RDD'>


In [29]:
# RDD.take(n): RDD의 상위 n개의 데이터(line) 반환
# RDD.first(): RDD의 첫번째 데이터(line) 반환
print(lines.take(5))
print(lines.first())
# take는 각 line 데이터를 포함한 list로 반환
# 각 line의 type은 str
print(type(lines.take(5)))
print(type(lines.first()))

['196\t242\t3\t881250949', '186\t302\t3\t891717742', '22\t377\t1\t878887116', '244\t51\t2\t880606923', '166\t346\t1\t886397596']
196	242	3	881250949
<class 'list'>
<class 'str'>


## 3. **Transformation** (MapReduce의 mapper와 유사)
- `RDD.map(func)`: 대표적인 Transformation 중 하나
> - RDD를 원하는 데이터의 형태로 변형함
> - `map`의 인자는 `RDD` 하나의 line을 전달받아서, 원하는 형태로 변형하여 반환하는 **함수**
> - 일반적으로 간단한 함수의 형태가 많으므로, `lambda` 함수가 자주 사용됨

- **목표**: 각 평점(rating)별 개수를 확인하고자 함
- **rating**을 제외한 나머지 데이터는 불필요하므로, `map`을 통해 `rating`만 반환하도록 함
- 하나의 값만 반환할 경우, **key** 없이 **value**로만 구성된 `RDD`가 됨
- `str.split(a)`: str을 a기준으로 split하여 분리된 값을 list로 반환

In [30]:
ratings = lines.map(lambda x:x.split('\t')[2])
# map의 함수 인자는 RDD(lines)의 각 line(str)이 됨
# rating은 lines를 분리했을 때, 세번째 요소이므로 index=2

In [31]:
ratings.take(10)

['3', '3', '1', '2', '1', '4', '2', '5', '3', '3']

In [15]:
print(type(lines))
print(type(ratings))
# SparkContext.textFile 또는 SparkContext.parallelize로 초기에 생성된 RDD의 타입은 RDD이며
# Transformation 및 Action 과정을 거친 RDD는 PipelinedRDD (RDD의 모든 기능을 사용할 수 있음)

<class 'pyspark.rdd.RDD'>
<class 'pyspark.rdd.PipelinedRDD'>


## 4. **Action** (MapReduce의 reducer와 유사)
- `RDD.collect()`: RDD를 모아서 list로 반환
- `RDD.countByValue()`: RDD value를 기준으로 count를 수행하며, 각 value에 대한 개수를 `dict type`으로 반환(value가 key, 각 value의 개수가 value)

In [16]:
lines.collect()[:10] # lines의 상위 10개 데이터 확인

['196\t242\t3\t881250949',
 '186\t302\t3\t891717742',
 '22\t377\t1\t878887116',
 '244\t51\t2\t880606923',
 '166\t346\t1\t886397596',
 '298\t474\t4\t884182806',
 '115\t265\t2\t881171488',
 '253\t465\t5\t891628467',
 '305\t451\t3\t886324817',
 '6\t86\t3\t883603013']

In [17]:
ratings.countByValue()
# ratings의 값인 1,2,3,4,5가 key
# 각 rating에 대한 개수가 value인 dict type을 반환

defaultdict(int, {'3': 27145, '1': 6110, '2': 11370, '4': 34174, '5': 21201})

In [18]:
# SparkContext 종료
# SparkContext를 종료하지 않은채로 동일한 이름의 SparkContext 생성을 시도하면 오류가 발생할 수 있음
spark.stop() 